# arXiv CS Corpus Collector — Sakhiur
**Categories:** cs.AI, cs.LG, cs.IR, cs.DB, cs.SE  
**Task:** 50k abstracts (shared pull) + 500 PDFs per category  
**No SQLite — flat files only**

In [12]:
!pip install requests tqdm lxml pandas


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports

In [13]:
import os, json, time, requests
from pathlib import Path
from tqdm import tqdm
from lxml import etree
from concurrent.futures import ThreadPoolExecutor


## Config 

In [14]:
CONFIG = {
    "member"      : "sakhiur",
    "categories"  : ["cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE"],
    "date_from"   : "2021-01-01",
    "date_until"  : "2026-05-31",
    "abstracts_target" : 25000,   # your half of the 50k corpus
    "pdfs_per_cat"     : 500,
    "pdf_workers" : 4,            # safe for i7, no GPU
    "oai_sleep"   : 20,           # seconds between OAI pages (be polite)
    "pdf_sleep"   : 1.0,          # seconds between PDF requests
}

# ── Directory layout ────────────────────────────────────────────
BASE        = Path("../data/arxiv")
PDF_DIR     = BASE / "pdfs"
META_DIR    = BASE / "metadata"

for d in [BASE, PDF_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Per-category PDF subdirs
for cat in CONFIG["categories"]:
    (PDF_DIR / cat).mkdir(exist_ok=True)

# Flat-file ledgers
ABSTRACTS_FILE  = BASE / "abstracts.jsonl"        # all metadata + abstracts
DONE_IDS_FILE   = BASE / "downloaded_ids.txt"     # successful PDF downloads
FAILED_IDS_FILE = BASE / "failed_ids.txt"         # failed PDF downloads
SEEN_IDS_FILE   = BASE / "seen_ids.txt"           # all harvested arxiv IDs (dedup)

print("Dirs ready. Config:")
print(json.dumps(CONFIG, indent=2))

Dirs ready. Config:
{
  "member": "sakhiur",
  "categories": [
    "cs.AI",
    "cs.LG",
    "cs.IR",
    "cs.DB",
    "cs.SE"
  ],
  "date_from": "2021-01-01",
  "date_until": "2026-05-31",
  "abstracts_target": 25000,
  "pdfs_per_cat": 500,
  "pdf_workers": 4,
  "oai_sleep": 20,
  "pdf_sleep": 1.0
}


## Ledger helpers (flat files)

In [24]:
def load_set(filepath):
    """Load a .txt ledger into a set. Returns empty set if file missing."""
    p = Path(filepath)
    if not p.exists():
        return set()
    # Use UTF-8 encoding to correctly read files with non-ASCII characters
    return set(p.read_text(encoding="utf-8").splitlines())

def append_id(filepath, arxiv_id):
    """Append a single ID to a ledger file."""
    with open(filepath, "a") as f:
        f.write(arxiv_id + "\n")

def append_abstract(record: dict):
    """Append one paper record to abstracts.jsonl."""
    with open(ABSTRACTS_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Pre-load existing state (safe to re-run after crash)
seen_ids   = load_set(SEEN_IDS_FILE)
done_ids   = load_set(DONE_IDS_FILE)
failed_ids = load_set(FAILED_IDS_FILE)

print(f"Resuming: {len(seen_ids)} seen | {len(done_ids)} PDFs done | {len(failed_ids)} failed")

Resuming: 25004 seen | 2491 PDFs done | 4 failed


## OAI-PMH parser

In [16]:
OAI_BASE = "https://export.arxiv.org/oai2"
NS = {
    "oai"   : "http://www.openarchives.org/OAI/2.0/",
    "arxiv" : "http://arxiv.org/OAI/arXiv/"
}

def fetch_oai_page(from_date, until_date, set_spec, token=None):
    if token:
        params = {"verb": "ListRecords", "resumptionToken": token}
    else:
        params = {
            "verb"           : "ListRecords",
            "metadataPrefix" : "arXiv",
            "set"            : set_spec,
            "from"           : from_date,
            "until"          : until_date,
        }
    resp = requests.get(OAI_BASE, params=params, timeout=60)
    resp.raise_for_status()
    return resp.content

def parse_oai_page(xml_bytes):
    root    = etree.fromstring(xml_bytes)
    records = []

    for record in root.findall(".//oai:record", NS):
        header = record.find("oai:header", NS)
        if header is not None and header.get("status") == "deleted":
            continue
        meta = record.find(".//arxiv:arXiv", NS)
        if meta is None:
            continue

        arxiv_id   = meta.findtext("arxiv:id",         namespaces=NS) or ""
        title      = (meta.findtext("arxiv:title",     namespaces=NS) or "").replace("\n", " ").strip()
        abstract   = (meta.findtext("arxiv:abstract",  namespaces=NS) or "").replace("\n", " ").strip()
        categories = (meta.findtext("arxiv:categories",namespaces=NS) or "").strip()
        submitted  = (meta.findtext("arxiv:created",   namespaces=NS) or "").strip()
        authors    = "; ".join(
            f"{a.findtext('arxiv:forenames', namespaces=NS) or ''} "
            f"{a.findtext('arxiv:keyname',   namespaces=NS) or ''}".strip()
            for a in meta.findall("arxiv:authors/arxiv:author", NS)
        )
        primary_cat = categories.split()[0] if categories else ""

        if arxiv_id:
            records.append({
                "id"          : arxiv_id,
                "title"       : title,
                "abstract"    : abstract,
                "authors"     : authors,
                "categories"  : categories,
                "primary_cat" : primary_cat,
                "submitted"   : submitted,
                "text"        : f"{title}. {abstract}"  # ready for embedding
            })

    token_el = root.find(".//oai:resumptionToken", NS)
    token    = token_el.text if token_el is not None and token_el.text else None
    return records, token

## Harvest abstracts

In [7]:
def harvest_abstracts():
    global seen_ids
    total_new = 0

    for cat in CONFIG["categories"]:
        print(f"\n── Harvesting {cat} ──")
        token   = None
        page    = 0
        cat_new = 0

        while True:
            try:
                xml             = fetch_oai_page(
                    CONFIG["date_from"], CONFIG["date_until"],
                    set_spec="cs", token=token
                )
                records, token  = parse_oai_page(xml)
            except Exception as e:
                print(f"  Error page {page}: {e} — retry in 30s")
                time.sleep(30)
                continue

            for rec in records:
                if rec["primary_cat"] != cat:
                    continue
                if rec["id"] in seen_ids:
                    continue

                append_abstract(rec)
                append_id(SEEN_IDS_FILE, rec["id"])
                seen_ids.add(rec["id"])
                cat_new   += 1
                total_new += 1

            page += 1
            print(f"  Page {page} | {cat}: {cat_new} | Total: {len(seen_ids)}", end="\r")

            if token is None and page > 150:
                break
            time.sleep(CONFIG["oai_sleep"])

        print(f"  {cat} done: {cat_new} papers")

    print(f"\nHarvest complete. {total_new} new abstracts. Total: {len(seen_ids)}")

harvest_abstracts()


── Harvesting cs.AI, cs.LG, cs.IR, cs.DB, cs.SE ──


KeyboardInterrupt: 

## Quick stats after harvest

In [17]:
from collections import Counter

cat_counts = Counter()
total      = 0

with open(ABSTRACTS_FILE, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        cat_counts[rec["primary_cat"]] += 1
        total += 1

print(f"Total abstracts: {total}")
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat:10s}: {count:,}")

Total abstracts: 25004
  cs.AI     : 7,369
  cs.DB     : 1,532
  cs.IR     : 3,076
  cs.LG     : 10,746
  cs.SE     : 2,281


## Build PDF download queue

In [18]:
# Pick 500 papers per category from the harvested abstracts.
# Sampling strategy: earliest 500 by submission date (reproducible).
# Change to random_sample=True for random selection.

import random

RANDOM_SAMPLE = False   # set True for random 500; False for earliest 500
SEED          = 42

def build_pdf_queue(pdfs_per_cat=500):
    by_cat = {cat: [] for cat in CONFIG["categories"]}

    with open(ABSTRACTS_FILE, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            cat = rec["primary_cat"]
            if cat in by_cat:
                by_cat[cat].append(rec["id"])

    queue = []
    for cat, ids in by_cat.items():
        if RANDOM_SAMPLE:
            random.seed(SEED)
            selected = random.sample(ids, min(pdfs_per_cat, len(ids)))
        else:
            selected = ids[:pdfs_per_cat]   # already in harvest order ≈ date order
        
        for arxiv_id in selected:
            queue.append((arxiv_id, cat))
        print(f"  {cat}: {len(selected)} queued")

    print(f"Total PDF queue: {len(queue)}")
    return queue

pdf_queue = build_pdf_queue(CONFIG["pdfs_per_cat"])

  cs.AI: 500 queued
  cs.LG: 500 queued
  cs.IR: 500 queued
  cs.DB: 500 queued
  cs.SE: 500 queued
Total PDF queue: 2500


## PDF downloader

In [19]:
# - Saves to pdfs/<category>/<arxiv_id>.pdf
# - Skips if file already exists (resume-safe)
# - Logs done/failed to flat files

PDF_BASE = "https://arxiv.org/pdf/"

def download_one(item):
    arxiv_id, cat = item
    safe_id  = arxiv_id.replace("/", "_")
    pdf_path = PDF_DIR / cat / f"{safe_id}.pdf"

    # Skip if already done or previously failed
    if pdf_path.exists() or arxiv_id in done_ids or arxiv_id in failed_ids:
        return "skip"

    url = f"{PDF_BASE}{arxiv_id}v1"
    try:
        resp = requests.get(url, timeout=45, stream=True)
        resp.raise_for_status()
        with open(pdf_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        append_id(DONE_IDS_FILE, arxiv_id)
        done_ids.add(arxiv_id)
        time.sleep(CONFIG["pdf_sleep"])
        return "ok"
    except Exception as e:
        append_id(FAILED_IDS_FILE, arxiv_id)
        failed_ids.add(arxiv_id)
        return f"fail:{e}"

def download_all_pdfs(queue):
    pending = [item for item in queue
               if item[0] not in done_ids and item[0] not in failed_ids]
    print(f"Pending: {len(pending)} | Already done: {len(done_ids)} | Failed: {len(failed_ids)}")

    results = {"ok": 0, "skip": 0, "fail": 0}
    with ThreadPoolExecutor(max_workers=CONFIG["pdf_workers"]) as ex:
        for res in tqdm(ex.map(download_one, pending), total=len(pending)):
            if res == "ok":   results["ok"]   += 1
            elif res == "skip": results["skip"] += 1
            else:               results["fail"] += 1

    print(f"Done: {results['ok']} | Skipped: {results['skip']} | Failed: {results['fail']}")

download_all_pdfs(pdf_queue)

Pending: 5 | Already done: 2491 | Failed: 4


100%|██████████| 5/5 [00:00<00:00, 9118.05it/s]

Done: 0 | Skipped: 5 | Failed: 0


## Retry failed downloads

In [26]:
# arXiv sometimes returns 403/503 transiently. One retry pass is usually enough.

def retry_failed(queue):
    global failed_ids
    retry_queue = [item for item in queue if item[0] in failed_ids]
    print(f"Retrying {len(retry_queue)} failed downloads...")

    # Clear from failed ledger before retry
    failed_ids_copy = set(failed_ids)
    for arxiv_id, _ in retry_queue:
        failed_ids.discard(arxiv_id)

    # Rewrite failed file without the retried IDs
    remaining_failed = failed_ids_copy - {item[0] for item in retry_queue}
    FAILED_IDS_FILE.write_text("\n".join(remaining_failed))

    download_all_pdfs(retry_queue)

retry_failed(pdf_queue)

Retrying 4 failed downloads...
Pending: 4 | Already done: 2491 | Failed: 0


100%|██████████| 4/4 [00:01<00:00,  3.09it/s]

Done: 0 | Skipped: 0 | Failed: 4


## Final summary

In [27]:
def final_summary():
    print("=" * 50)
    print("FINAL SUMMARY")
    print("=" * 50)

    # Abstracts
    n_abstracts = sum(1 for _ in open(ABSTRACTS_FILE, encoding="utf-8"))
    abs_size_mb = ABSTRACTS_FILE.stat().st_size / 1e6
    print(f"\nAbstracts : {n_abstracts:,} records | {abs_size_mb:.1f} MB")

    # PDFs per category
    print("\nPDFs per category:")
    total_pdf_size = 0
    for cat in CONFIG["categories"]:
        cat_dir   = PDF_DIR / cat
        pdfs      = list(cat_dir.glob("*.pdf"))
        size_mb   = sum(p.stat().st_size for p in pdfs) / 1e6
        total_pdf_size += size_mb
        print(f"  {cat:10s}: {len(pdfs):4d} PDFs | {size_mb:.1f} MB")

    print(f"\nTotal PDF size : {total_pdf_size/1024:.2f} GB")
    print(f"Failed downloads: {len(load_set(FAILED_IDS_FILE))}")
    print("=" * 50)

final_summary()

FINAL SUMMARY

Abstracts : 25,004 records | 73.2 MB

PDFs per category:
  cs.AI     :  498 PDFs | 924.1 MB
  cs.LG     :  500 PDFs | 1613.3 MB
  cs.IR     :  500 PDFs | 888.6 MB
  cs.DB     :  498 PDFs | 925.8 MB
  cs.SE     :  500 PDFs | 741.9 MB

Total PDF size : 4.97 GB
Failed downloads: 4
